In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

In [ ]:
from matplotlib.colors import LogNorm

In [ ]:
import ast

In [ ]:
plt.rcParams.update({'font.size': 18})

In [ ]:
FNAME = 'pmt_waveforms.csv'
wfdf = pd.read_csv(FNAME)

In [ ]:
wfdf = pd.read_csv('pmt_waveforms.csv',\
                   converters={'wf_00': ast.literal_eval,'wf_01': ast.literal_eval,'wf_02': ast.literal_eval})

In [ ]:
print(wfdf['wf_00'][0][0])

In [ ]:
print (wfdf.keys())
print (wfdf.shape)

In [ ]:
# import pulse-finding algorithms
import pulsefinding

### Find SPEs

In [ ]:
ctr = 0

SPE_wf_collection_v = []
# initialize an SPE kernel of 150 time-ticks
SPE_wf_KERNEL_v = np.zeros(150)
SPECTR = 0

for idx,row in wfdf.iterrows():
    
    # select how many SPEs to save
    if (SPECTR >= 2000): break
    
    
    if (SPECTR%100 == 0): print('Found %i SPEs so far...'%SPECTR)
    
    wf00 = np.array(row['wf_00'])
    
    try:
        # run pulse-finding algorithm
        base, rms, spectr, SPE_ampl_v, SPE_tick_v, SPE_wf_v = pulsefinding.GetSPE(wf00,DEBUG=False)
    # catch an excepton if the algorithm fails. Print out the error message
    except ValueError as e:
        print('caught error: ',e)
        continue

    for SPE_wf in SPE_wf_v:
        
        # normalize recorded SPEs to build the kernel
        spe_wf = np.array(SPE_wf)
        spe_wf /= np.max(spe_wf)
        SPE_wf_collection_v.append(spe_wf)
        SPE_wf_KERNEL_v += spe_wf
        # keep track of how many SPEs we found
        SPECTR += 1
        
    ctr += 1
    
    # plot a few waveforms and SPEs to make sure things are looking good
    if (ctr < 10):
        fig = plt.figure(figsize=(14,4))
        plt.plot(wf00,'k--')
        for SPEtick in SPE_tick_v:
            plt.axvline(SPEtick)
        plt.xlabel('Time Tick [64 MHz]')
        plt.ylabel('ADC')
        plt.title('PMT 00 Waveform',fontsize=14)
        plt.show()

### Plot the SPE Kernel

In [ ]:
fig = plt.figure(figsize=(10,6))

SPE_WF_KERNEL_NORM_v = SPE_wf_KERNEL_v/SPECTR

for wf in SPE_wf_collection_v:
    plt.plot(wf,'r--',lw=1,alpha=0.5)
plt.plot(SPE_WF_KERNEL_NORM_v,'k--')
plt.xlabel('Tmie Tick [64 MHz]')
plt.ylabel('Amplitude (re)')
plt.show()

In [ ]:
# Pad the waveform so that it has the same length as the PMT waveform

SPE_WF_KERNEL_NORM_v_PAD = np.pad(SPE_WF_KERNEL_NORM_v,(0,1500-len(SPE_WF_KERNEL_NORM_v)))
print(len(SPE_WF_KERNEL_NORM_v_PAD))

SPE_wf_collection_v_PAD = []
for wf in SPE_wf_collection_v:
    wf_PAD = np.pad(wf,(0,1500-len(wf)))
    SPE_wf_collection_v_PAD.append(wf_PAD)

In [ ]:
# check the output of the padded waveform
fig = plt.figure(figsize=(10,6))

for wf in SPE_wf_collection_v_PAD:
    plt.plot(wf,'r--',lw=1,alpha=0.5)
plt.plot(SPE_WF_KERNEL_NORM_v_PAD,'k--')
#plt.xlim([0,100])
plt.show()

In [ ]:
from numpy.fft import fft, ifft, ifftshift, fftfreq

In [ ]:
def GetFFT(waveform,FREQ):
    
    fft_result = fft(waveform)
    magnitude = abs(fft_result)          # Complex → magnitude
    freqs = fftfreq(len(waveform), d=1./(FREQ))  # Frequency axis in MHz

    # Keep only positive frequencies (the spectrum is mirrored)
    pos_mask = freqs >= 0
    freqs = freqs[pos_mask]
    magnitude = magnitude[pos_mask]

    # Normalize 
    magnitude = magnitude / len(waveform)     # Scale by N
    
    return freqs,magnitude

In [ ]:
# Compute FFT

freqs_KERNEL,magnitude_KERNEL = GetFFT(SPE_WF_KERNEL_NORM_v_PAD,64.)

freqs_SPEs = np.zeros(len(freqs_KERNEL))
magnitude_SPEs = np.zeros(len(magnitude_KERNEL))

ctr = 0
for wf in SPE_wf_collection_v_PAD:
    f_SPEs,m_SPEs = GetFFT(wf,64.)
    freqs_SPEs = f_SPEs
    magnitude_SPEs += m_SPEs
    ctr += 1
    
magnitude_SPEs /= float(ctr)


magnitude_NOISE = magnitude_SPEs - magnitude_KERNEL

In [ ]:
# Plot
plt.figure(figsize=(10, 6))
plt.plot(freqs_KERNEL, magnitude_KERNEL,'k--',label='KERNEL')
plt.plot(freqs_SPEs, magnitude_SPEs,'r--',label='SPEs')
plt.plot(freqs_SPEs, magnitude_NOISE,'b--',label='Noise')
plt.xlabel("Frequency (MHz)")
plt.ylabel("Magnitude")
plt.title("FFT Magnitude Spectrum")
plt.legend(loc=1)
plt.grid(True)
plt.tight_layout()
plt.yscale('log')
#plt.xscale('log')
plt.show()

In [ ]:
magnitude_SNR = magnitude_KERNEL / magnitude_NOISE

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(freqs_KERNEL, magnitude_SNR,'k--')
plt.xlabel("Frequency (MHz)")
plt.ylabel("Signal-To-Noise")
plt.grid(True)
plt.tight_layout()
plt.yscale('log')
#plt.xscale('log')
plt.show()

In [ ]:
print(magnitude_SNR)

In [ ]:
fout = open('spenoisemag00.txt','w')
for val in magnitude_NOISE:
    fout.write('%g,'%val)
fout.close()
    
fout = open('snrmag00.txt','w')
for val in magnitude_SNR:
    fout.write('%g,'%val)
fout.close()

In [ ]:
ctr = 0
for idx,row in wfdf.iterrows():
    if (idx%100 != 0): continue
    fig = plt.figure(figsize=(15,6))
    print('row has run number: ',row['run'])
    for pmt in range(0,3):
        plt.plot(np.array(row['wf_%02i'%pmt])+pmt*10)
    plt.xlabel('Time Tick [15.625 ns / 64 MHz]',fontsize=16)
    plt.ylabel('ADC',fontsize=16)
    plt.title(r'23.4 $\mu$s unbiased MicroBooNE PMT readout waveform')
    plt.tight_layout()
    plt.show()
    ctr += 1
    if (ctr >= 10):
        break

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.hist(wfdf['event'].values,bins=100,histtype='step',lw=2)
plt.xlabel('Event Number')
plt.ylabel('Entries')
plt.show()

In [ ]:
wfdfsub = wfdf.query('event*event < 2000')

print(wfdf.keys())
print(wfdfsub.shape)
#print(wfdfsub)

In [ ]:
print(wfdfsub['event'].values)

In [ ]:
def wfsum(x):
    return np.add(x['wf_00'],x['wf_01'])

In [ ]:
wfdf['wf_sum'] = wfdf.apply(lambda x: wfsum(x), axis=1)

In [ ]:
ctr = 0
for idx,row in wfdf.iterrows():
    if (idx%100 != 0): continue
    fig = plt.figure(figsize=(15,6))
    plt.plot(np.array(row['wf_sum']),color='k')
    plt.xlabel('Time Tick [15.625 ns / 64 MHz]',fontsize=16)
    plt.ylabel('ADC',fontsize=16)
    plt.title(r'23.4 $\mu$s unbiased MicroBooNE PMT readout waveform')
    plt.tight_layout()
    plt.show()
    ctr += 1
    if (ctr >= 10):
        break

In [ ]:
def wfmax(x):
    return np.max(x['wf_00'])

In [ ]:
wfdf['wf_00_max'] = wfdf.apply(lambda x: wfmax(x), axis=1)

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.hist(wfdf['wf_00_max'].values,bins=100,histtype='step',lw=2)
plt.xlabel('max adc on waveform')
plt.ylabel('Entries')
plt.yscale('log')
plt.show()

In [ ]:
dflarge = wfdf.query('wf_00_max > 2100')

In [ ]:
ctr = 0
for idx,row in dflarge.iterrows():
    fig = plt.figure(figsize=(15,6))
    for pmt in range(0,3):
        plt.plot(np.array(row['wf_%02i'%pmt])+pmt*10)
    plt.xlabel('Time Tick [15.625 ns / 64 MHz]',fontsize=16)
    plt.ylabel('ADC',fontsize=16)
    plt.title(r'23.4 $\mu$s unbiased MicroBooNE PMT readout waveform')
    plt.tight_layout()
    plt.show()
    ctr += 1
    if (ctr >= 10):
        break